# Uni-MuMER cải tiến - dev-caitien / Kaggle 2xT4

QLoRA 1 epoch, ưu tiên mẫu khó có giới hạn. Validation vẫn dùng loss chuẩn.
Cùng dữ liệu, split và thông số baseline dev_tt; không dùng checkpoint cũ.


In [ ]:
# 1. Cấu hình
import json
import os
import uuid

RUN_TRAINING = False  # Đổi thành True khi đã sẵn sàng train và ghi DagsHub.
RUN_UUID = globals().get("RUN_UUID") or uuid.uuid4().hex

PROJECT_DIR = "/kaggle/working/test-unimer"
CONDA_DIR = "/kaggle/working/miniconda"
ENV_DIR = f"{CONDA_DIR}/envs/unimumer"
PYTHON = f"{ENV_DIR}/bin/python"

RUN_DIR = f"/kaggle/working/runs/{RUN_UUID}"
BASE_YAML_CONFIG = "train/Uni-MuMER-train.yaml"
RUNTIME_YAML_CONFIG = f"{RUN_DIR}/runtime.yaml"
OUTPUT_DIR = f"{RUN_DIR}/checkpoints"
SYNC_STATE_PATH = f"{RUN_DIR}/checkpoint_sync_state.json"
BASELINE_DATA_ROOT = "/kaggle/working/baseline-data"
NOTEBOOK_PATH = f"{PROJECT_DIR}/uni-mumer-kaggle-dagshub-banmoi.ipynb"

DAGSHUB_USERNAME = "NhatPot"
EXPERIMENT_NAME = "Uni-MuMER-Qwen2.5-VL-3B"
DATASET_REVISION = "8fcc66de1298524c9e47f542a726090c44532e6e"
EXPECTED_SPLIT_SHA256 = "a3ffe3480ceaafcaebf96f31102e9f97dadd9effc56ca235dc573cb0ff00a11a"

# Các cell Bash dùng chung môi trường Python riêng.
os.environ.pop("PYTHONHOME", None)
os.environ.pop("MLFLOW_TRACKING_PASSWORD", None)
os.environ.update({
    key: str(value) for key, value in {
        "PROJECT_DIR": PROJECT_DIR, "CONDA_DIR": CONDA_DIR,
        "ENV_DIR": ENV_DIR, "PYTHON": PYTHON, "RUN_DIR": RUN_DIR,
        "RUN_UUID": RUN_UUID, "RUN_TRAINING": int(RUN_TRAINING),
        "BASE_YAML_CONFIG": BASE_YAML_CONFIG, "RUNTIME_YAML_CONFIG": RUNTIME_YAML_CONFIG,
        "OUTPUT_DIR": OUTPUT_DIR, "SYNC_STATE_PATH": SYNC_STATE_PATH,
        "NOTEBOOK_PATH": NOTEBOOK_PATH, "YAML_CONFIG": "",
        "PYTHONNOUSERSITE": 1, "PYTHONPATH": PROJECT_DIR,
        "MLFLOW_TRACKING_URI": f"https://dagshub.com/{DAGSHUB_USERNAME}/test-unimer.mlflow",
        "MLFLOW_TRACKING_USERNAME": DAGSHUB_USERNAME,
        "EXPERIMENT_VARIANT": "improved",
        "MLFLOW_EXPERIMENT_NAME": EXPERIMENT_NAME, "MLFLOW_FLATTEN_PARAMS": "TRUE",
        "MLFLOW_TAGS": json.dumps({"run_uuid": RUN_UUID, "source": "kaggle", "task": "sft", "variant": "improved8000"}),
    }.items()
})
env_bin = f"{ENV_DIR}/bin"
os.environ["PATH"] = os.pathsep.join(
    [env_bin] + [p for p in os.environ.get("PATH", "").split(os.pathsep) if p != env_bin]
)

if RUN_TRAINING:
    from kaggle_secrets import UserSecretsClient
    os.environ["MLFLOW_TRACKING_PASSWORD"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

print("Run:", RUN_UUID)
print("Output:", OUTPUT_DIR)


In [ ]:
%%bash
# 2. Tạo môi trường Python 3.10
set -euo pipefail

if [[ ! -x "$CONDA_DIR/bin/conda" ]]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -p "$CONDA_DIR"
  rm -f /tmp/miniconda.sh
fi

if [[ ! -x "$PYTHON" ]]; then
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
  "$CONDA_DIR/bin/conda" create -n unimumer python=3.10 -y
fi

"$PYTHON" --version


In [ ]:
%%bash
# 3. Lấy source cải tiến dev-caitien
set -euo pipefail

if [[ ! -d "$PROJECT_DIR/.git" ]]; then
  git clone --branch dev-caitien --single-branch https://github.com/NhatPot/test-unimer.git "$PROJECT_DIR"
fi
[[ "$(git -C "$PROJECT_DIR" branch --show-current)" == dev-caitien ]] \
  || { echo "Repo hiện có không ở dev-caitien; giữ nguyên, chọn đúng checkout trước."; exit 1; }
git -C "$PROJECT_DIR" rev-parse --short HEAD


In [ ]:
# %%bash
# # 3.5 Download test data
# set -e

# cd "$PROJECT_DIR"

# pip install -q gdown

# gdown "1uHT7iOFHGASc9_HB-PhpIuJizlOUOg1L" -O crohme_test_images.zip

# unzip -q crohme_test_images.zip -d "$PROJECT_DIR"

# rm crohme_test_images.zip

In [ ]:
%%bash
# 4. Cài dependency trong Python riêng
set -euo pipefail
cd "$PROJECT_DIR"

"$PYTHON" --version
"$PYTHON" -m pip install -q -r requirements.txt -e train/LLaMA-Factory
"$PYTHON" -m pip check
"$PYTHON" - <<'PY'
import torch
assert torch.cuda.is_available(), "Chưa nhận CUDA"
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPU:", names)
assert len(names) == 2 and all("T4" in name for name in names), "Cấu hình này dành cho 2 GPU T4"
PY


In [ ]:
# 5. Thông số cải tiến: chỉnh tại đây, không cần sửa YAML gốc.
import subprocess

os.environ["YAML_CONFIG"] = ""
YAML_OVERRIDES = {
    "dataset": "parquet_crohme_train, parquet_crohme_train_can, parquet_crohme_train_tree, parquet_crohme_train_error_find, parquet_crohme_train_error_fix, parquet_hme100k_train",
    "max_samples": 8000,  # Giới hạn mỗi nguồn TRƯỚC khi chia tập.
    "val_size": 0.1,      # 10% nhóm ảnh; source không chia lại theo dòng.
    "seed": 42,
    "model_revision": "66285546d2b821cf421d4f5eb2576359d3770cd3",
    "use_hard_example_loss": True,
    "hard_example_loss_alpha": 0.5,
    "hard_example_loss_max_weight": 1.5,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 64,
    "learning_rate": 1.0e-4,
    "lora_rank": 64,
    "bf16": False,
    "fp16": True,
    "logging_steps": 1,
    "save_steps": 50,
    "eval_steps": 50,
    "save_total_limit": 2,
    "save_only_model": False,
    "load_best_model_at_end": False,  # Eval sau train đo đúng trọng số CUỐI.
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "output_dir": OUTPUT_DIR,
    "overwrite_output_dir": False,
}
DATASET_DIR = f"{BASELINE_DATA_ROOT}/n{YAML_OVERRIDES['max_samples']}-v{YAML_OVERRIDES['val_size']}-s{YAML_OVERRIDES['seed']}"
result = subprocess.run(
    [PYTHON, "-m", "scripts.prepare_baseline_data",
     "--project-dir", PROJECT_DIR, "--output-dir", DATASET_DIR,
     "--base-yaml", BASE_YAML_CONFIG, "--runtime-yaml", RUNTIME_YAML_CONFIG,
     "--dataset-revision", DATASET_REVISION,
     "--expected-manifest-sha256", EXPECTED_SPLIT_SHA256,
     "--overrides-json", json.dumps(YAML_OVERRIDES)],
    cwd=PROJECT_DIR, capture_output=True, text=True,
)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Không chuẩn bị được dữ liệu/YAML; xem lỗi phía trên.")
config = json.loads(result.stdout)
YAML_CONFIG, OUTPUT_DIR = config["yaml_config"], config["output_dir"]
tags = json.loads(os.environ["MLFLOW_TAGS"])
tags.update(variant="improved8000", split_sha256=config["manifest_sha256"])
os.environ.update(YAML_CONFIG=YAML_CONFIG, OUTPUT_DIR=OUTPUT_DIR,
                  DATASET_DIR=DATASET_DIR, SPLIT_MANIFEST=config["manifest_path"],
                  MLFLOW_TAGS=json.dumps(tags))
print(json.dumps({"samples": config["counts"], "config": config["summary"]}, indent=2))


In [ ]:
%%bash
# 6. Kiểm tra quyền ghi DagsHub (tạo rồi xóa một run thử)
set -euo pipefail
[[ "$RUN_TRAINING" == 1 ]] || { echo "Bỏ qua DagsHub: RUN_TRAINING=False"; exit 0; }
cd "$PROJECT_DIR"
"$PYTHON" scripts/dagshub_logger.py check --experiment "$MLFLOW_EXPERIMENT_NAME"


In [ ]:
%%bash
# 7. Training + sync checkpoint bằng wrapper cũ
set -euo pipefail
[[ "$RUN_TRAINING" == 1 ]] || { echo "Bỏ qua train: RUN_TRAINING=False"; exit 0; }
cd "$PROJECT_DIR"
test -n "$YAML_CONFIG"
[[ ! -e "$OUTPUT_DIR" ]] || { echo "Output đã tồn tại; dùng run mới để tránh ghi đè."; exit 1; }

MPLBACKEND=Agg "$PYTHON" scripts/train_with_checkpoint_sync.py \
  --experiment "$MLFLOW_EXPERIMENT_NAME" \
  --run-uuid "$RUN_UUID" \
  --yaml-config "$YAML_CONFIG" \
  --output-dir "$OUTPUT_DIR" \
  --project-dir "$PROJECT_DIR" \
  --run-name "uni-mumer-$RUN_UUID" \
  --state-path "$SYNC_STATE_PATH" \
  --interval-seconds 180 \
  --stable-seconds 20 \
  --artifact-path checkpoints


In [ ]:
# %%bash
# # 8. Full Benchmark Test (3 CROHME datasets)
# set -euo pipefail
# cd "$PROJECT_DIR"
# source "$CONDA_DIR/bin/activate" unimumer

# # Tìm checkpoint cuối cùng
# LAST_CKPT=$(ls -td "$OUTPUT_DIR"/checkpoint-* 2>/dev/null | head -1)

# if [[ -z "$LAST_CKPT" ]]; then
#   echo "ERROR: No checkpoint found in $OUTPUT_DIR"
#   exit 1
# fi

# echo "Testing with checkpoint: $LAST_CKPT"
# echo "Running full benchmark on 3 CROHME datasets (3,332 samples)..."
# echo ""

# # Chạy full benchmark
# python scripts/kaggle_full_test.py \
#   --base-model Qwen/Qwen2.5-VL-3B-Instruct \
#   --adapter-path "$LAST_CKPT" \
#   --test-datasets crohme_2014 crohme_2016 crohme_2019 \
#   --backup-dir example_data/backup \
#   --base-results-dir example_data/CROHME/results \
#   --output-dir kaggle_test_results \
#   --project-dir "$PROJECT_DIR" \
#   --batch-size 2

# # In summary
# echo ""
# echo "============================================================"
# echo "                    TEST SUMMARY"
# echo "============================================================"
# for dataset in crohme_2014 crohme_2016 crohme_2019; do
#   echo ""
#   echo "=== $dataset ==="
#   if [[ -f "kaggle_test_results/${dataset}_results.txt" ]]; then
#     cat "kaggle_test_results/${dataset}_results.txt" | grep -E "(Mean Edit Score|BLEU-4|Character Error Rate|Exact Match)" | head -4
#   else
#     echo "Results not found"
#   fi
# done

# echo ""
# echo "============================================================"
# echo "Full comparison table:"
# cat kaggle_test_results/comparison_table.txt

In [ ]:
%%bash
# 9. Upload checkpoint và cấu hình của run này
set -euo pipefail
[[ "$RUN_TRAINING" == 1 ]] || { echo "Bỏ qua upload: RUN_TRAINING=False"; exit 0; }
cd "$PROJECT_DIR"
test -n "$YAML_CONFIG"

"$PYTHON" scripts/dagshub_logger.py upload \
  --experiment "$MLFLOW_EXPERIMENT_NAME" \
  --run-uuid "$RUN_UUID" \
  --config "$YAML_CONFIG" \
  --output-dir "$OUTPUT_DIR" \
  --project-dir "$PROJECT_DIR" \
  --notebook "$NOTEBOOK_PATH"


In [ ]:
# %%bash
# cd /kaggle/working/test-unimer

# echo "Đang ở:"
# pwd

# echo "Branch hiện tại:"
# git branch --show-current

# echo "Trạng thái trước khi pull:"
# git status --short

# echo "Pull code mới nhất:"
# git pull